In [ ]:
# %pip install plotly

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [ ]:
df=pd.read_csv("loan.csv")

### 1. Data Understanding

In [ ]:
df.head()

In [ ]:
df.sample()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

### 2. Initial Data Quality Assessment

In [ ]:
missing_value_summary=pd.DataFrame({'missing_count':df.isnull().sum(),'missing_pct':df.isnull().mean()*100})
missing_value_summary=missing_value_summary.sort_values(by='missing_pct', ascending=False).reset_index()
display(missing_value_summary)

In [ ]:
# Check duplicate rows
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
# Identify columns containing only one unique value
constant_cols = df.columns[df.nunique(dropna=False) == 1].tolist()
print("Constant columns:", constant_cols)

### 3. Data Cleaning and Preparation

In [ ]:
# Remove constant columns as well as irrelevant columns
cols_to_drop = ['id', 'member_id', 'url', 'policy_code', 'desc']

df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [ ]:
# Convett Date columns
date_cols = ['issue_d', 'earliest_cr_line','last_pymnt_d', 'next_pymnt_d','last_credit_pull_d','hardship_start_date', 'hardship_end_date','payment_plan_start_date','debt_settlement_flag_date', 'settlement_date']

for col in date_cols:
    if col in df.columns:
        if not pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = pd.to_datetime(df[col],format='%b-%Y', errors='coerce')

In [ ]:
# Convert term to integer
df['term'] = pd.to_numeric(df['term'].astype('string').str.strip().str.replace(' months', '', regex=False), errors='coerce').astype('Int64')

**Handle missing values by category**

Instead of 75 individual checks, we'll use one consistent policy.

Feature category	                       Action	                   Reason

* Joint application fields	               Keep NaN	                   Not applicable to individual loans
  
* Hardship and settlement fields	       Keep NaN	                   Not applicable to most loans

* Historical credit metrics (~38% missing) Keep NaN	                   Largely unavailable before 2016

* Months-since-delinquency fields	       Keep NaN	                   Absence may reflect no recorded event
 
* emp_length, emp_title	                   Label missing as Unknown	   Categorical information

* annual_inc, dti, other numeric fields	   Preserve missing initially  Avoid introducing artificial values

* next_pymnt_d	                           Keep NaT	                   No scheduled payment date may be meaningful

In [ ]:
for col in ['grade', 'home_ownership', 'verification_status', 'application_type']:
    print(col, df[col].value_counts(dropna=False))

In [ ]:
# Fill missing employment categories
categorical_fill = ['emp_length', 'emp_title']

for col in categorical_fill:
    df[col] = df[col].fillna('Unknown')

# Observation: I am not filling every numerical missing value because EDA can handle missing observations, and arbitrary median imputation may distort the loan distributions which we're trying to understand.

In [ ]:
df[['annual_inc', 'dti']].quantile([0.25, 0.5, 0.75, 0.95, 0.99, 0.999])

In [ ]:
df.loc[df['dti'] < 0, 'dti'] = np.nan

In [ ]:
income_limit = df['annual_inc'].quantile(0.999)

print("Income threshold:", income_limit)
print("Records above threshold:", (df['annual_inc'] > income_limit).sum())

In [ ]:
fig = px.histogram(df[df['annual_inc'] <= income_limit],x='annual_inc', nbins=10,title='Annual Income Distribution')
fig.show()

* Annual income is right-skewed, with a median of $65,000.
* Only 2,165 records have annual income above $600,000, approximately 0.096% of the dataset.
* DTI also has extreme observations. Its 99.9th percentile is 106.63.
* These values are statistical outliers, not necessarily data errors.

In [ ]:
# winsorization (upper-tail capping) rather than row removal. It retains all observations while limiting the influence of extreme values.

df['annual_inc_capped'] = df['annual_inc'].clip(upper=600000)

df['dti_capped'] = df['dti'].clip(upper=106.63)

In [ ]:
df[['annual_inc_capped', 'dti_capped']].describe()

**4. Post-Cleaning Data Validation**

In [ ]:
# Columns where negative amounts should not occur
nonnegative_cols = ['loan_amnt','funded_amnt','installment','annual_inc','recoveries','collection_recovery_fee','total_pymnt','total_rec_prncp','out_prncp']

quality_checks = pd.DataFrame({'negative_values':[(df[col] < 0).sum() for col in nonnegative_cols],'missing_values':[df[col].isna().sum() for col in nonnegative_cols]},index=nonnegative_cols)

display(quality_checks)

In [ ]:
print("Loan term:")
print(df['term'].value_counts(dropna=False))

print("\nLoan status:")
print(df['loan_status'].value_counts(dropna=False))

print("\nLoan grades:")
print(df['grade'].value_counts(dropna=False))

In [ ]:
print("Final dataset shape:", df.shape)

print("Duplicate rows:", df.duplicated().sum())

print("Total missing cells:", df.isna().sum().sum())

print("Memory usage (MB):",round(df.memory_usage(deep=True).sum() / 1024**2, 2))

display(df.dtypes.value_counts())

# Observation: Since, 99.4 million values are missing, so we cant remove them as many belong to joint application, hardship, settlement, and historically unavailable credit fields.

In [ ]:
print("Funded amount exceeds loan amount:", (df['funded_amnt'] > df['loan_amnt']).sum())

print("Invalid interest rates:", ((df['int_rate'] < 0) | (df['int_rate'] > 100)).sum())

print("Principal Received > Funded:", (df['total_rec_prncp'] > df['funded_amnt'] + 0.01).sum())

In [ ]:
principal_issue = df[df['total_rec_prncp'] > df['funded_amnt'] + 0.01].copy()

principal_issue['difference'] = principal_issue['total_rec_prncp'] - principal_issue['funded_amnt']

display(principal_issue[['loan_status', 'funded_amnt', 'total_rec_prncp', 'difference', 'recoveries']])

In [ ]:
principal_issue['difference'].describe()

**Financial Data Quality Validation**
Identified 21 loans where recorded principal repayments exceeded funded amounts by $0.02–$0.25. All affected loans were classified as fully paid and had zero recorded recoveries. The discrepancies were retained and documented as minor financial reconciliation exceptions, potentially attributable to rounding or payment allocation differences. No records were removed.

Saving the cleaned DataFrame because this dataset contains 2.26 million rows and 140 columns, occupying approximately 4.5 GB of RAM and Without saving the prepared dataset, every time after restarting Jupyter, we would need to repeat entire data cleaning operations. But, with Pickle, we perform these steps once and save the result in my directory.

In [ ]:
df.to_pickle("Dataset_prepared.pkl")

**5. Exploratory Data Analysis (EDA)**

1. Loan Status Distribution

In [ ]:
# Number of loans in each category
loan_status_df = df['loan_status'].value_counts().reset_index()

# Percentage of loans in each category
loan_status_df['percentage'] = (loan_status_df['count'] / loan_status_df['count'].sum() * 100).round(2)
loan_status_df

In [ ]:
fig1 = px.bar(loan_status_df, x='count', y='loan_status', orientation='h',hover_data=['percentage'], title='Distribution of Loan Status')

fig1.update_layout(yaxis={'categoryorder': 'total ascending'})
fig1.show()

Observation 1: Loan Status Distribution

The portfolio is dominated by Fully Paid (46.09%) and Current (40.68%) loans. Charged Off loans account for approximately 11.6% of all records, representing a substantial segment for further recovery analysis. Since many loans remain active, the observed charge-off proportion should not be interpreted as the final default rate.

2. Loan Amount Distribution

In [ ]:
pd.options.display.float_format = '{:,.2f}'.format

In [ ]:
fig2 = px.histogram(df, x='loan_amnt', nbins=10, title='Distribution of Loan Amounts')
fig2.show()

df['loan_amnt'].describe()

Observation 2: Loan Amount Distribution

The average loan amount is approximately 15,047, while the median is 12,900, suggesting a right-skewed distribution. Half of the loans range between 8,000  and  20,000, indicating that loan amounts are concentrated in the lower to middle portion of the observed range.

3. Loan Amount by Loan Status

In [ ]:
fig3 = px.box(df, x='loan_amnt', y='loan_status', title='Loan Amount by Loan Status')
fig3.show()

Observation 3: Loan Amount by Loan Status
* Charged Off and Fully Paid: Their loan amount distributions appear broadly similar, with median loan amounts around $13K–$15K.
* Late loans: Loans classified as Late (31–120 days) appear to have slightly higher median loan amounts.
* Variation: Most major loan-status categories include loans ranging from approximately 500 to 40,000.
* Credit policy exceptions: These two categories appear to have lower loan amounts than most other groups.

4. Interest Rate Distribution

In [ ]:
fig4 = px.histogram(df, x='int_rate', nbins=10, title='Distribution of Interest Rates')
fig4.show()

df['int_rate'].describe()

Observation 4:

* Highest concentration: Most loans have interest rates between approximately 5% and 20%.
* Most frequent range: The 10%–15% interval contains the largest number of loans.
* Right-skewed distribution: The number of loans generally decreases as interest rates increase beyond 15%.
* High-interest loans: Relatively few loans have interest rates above 25%.

Business interpretation: This portfolio contains substantially more low-to-moderate-interest loans than high-interest loans. However, we cannot determine credit risk or recovery performance from interest rates alone.

5. Interest Rate by Loan Grade

In [ ]:
fig5 = px.box(df, x='int_rate', y='grade',
              category_orders={'grade': list('ABCDEFG')},
              title='Interest Rate by Loan Grade')
fig5.show()

Observation 5:

* Grade A: Lowest interest rates, with a median around 7%.
* Grades B and C: Median interest rates increase to approximately 11% and 14%.
* Grades D and E: Interest rates increase further, with medians around 18% and 21%.
* Grades F and G: Highest interest rates, with Grade G approaching 28%.

Business interpretation: Interest rates increase consistently from Grade A to Grade G, reflecting Organization's risk-based loan pricing. Higher-risk grades generally carry higher interest rates.

6. Loan Grade vs. Charge-Off Rate

In [ ]:
df['is_charged_off'] = df['loan_status'].str.contains('Charged Off', na=False)

chargeoff_by_grade = (df.groupby('grade')['is_charged_off'].mean() * 100).round(2)

fig = px.bar(chargeoff_by_grade.reset_index(), x='grade', y='is_charged_off', title='Observed Charge-Off Percentage by Loan Grade')
fig.show()

Observation 6:

* Grade A has the lowest observed charge-off percentage.
* Grades F and G have the highest, with more than one-third of loans classified as charged off.
* This supports the relationship between orgabization's risk grades and observed loan performance.

### Recovery on Charged-Off Loans

In [ ]:
# Creating a new DataFrame for Chargedoff Loans

charged_off_df = df[df['is_charged_off']].copy()


print("Charged-off loans:", len(charged_off_df))

charged_off_df.head()

In [ ]:
charged_off_df.shape

1. Recovery in Chargedoff accounts

In [ ]:
# How many charged-off loans have recovered any money.
charged_off_df['has_recovery'] = charged_off_df['recoveries'] > 0

recovery_df = charged_off_df['has_recovery'].value_counts().reset_index()
recovery_df['percentage'] = (recovery_df['count'] / recovery_df['count'].sum() * 100).round(2)

fig = px.bar(recovery_df, x='has_recovery', y='count', hover_data=['percentage'], title='Recovery in Charged-Off Loans')
fig.show()

In [ ]:
charged_off_df['has_recovery'].value_counts(normalize=True).mul(100).round(2)

Observation:

* Approximately 175K–180K charged-off loans have positive recoveries.
* Approximately 85K charged-off loans have zero recorded recoveries.
* Roughly two-thirds of charged-off loans have some recovery.

Business interpretation: A significant proportion of charged-off loans still generate recoveries. Charge-off does not necessarily mean the lender recovers nothing. We still cannot say that having a positive recovery means the recovered amount is financially significant.

2. Recovery Amount Distribution

In [ ]:
charged_off_df['recoveries'].describe()

In [ ]:
fig = px.histogram(charged_off_df[charged_off_df['recoveries'] > 0], x='recoveries', nbins=10, title='Distribution of Positive Recovery Amounts')
fig.show()

Observation:

* Strong right skew: The mean 1,172 is substantially greater than the median 549, indicating that some large recoveries pull the average upward.

* Low typical recovery: Half of all charged-off loans have recovered 549.37 or less.

* Recovery concentration: The histogram shows that most positive recoveries are concentrated below approximately 5,000, with relatively few large recoveries.

* High variability: The maximum recovery is approximately $39,860, much higher than the typical recovery.

3. Recovery Amount by Loan Grade

In [ ]:
recovery_by_grade = charged_off_df.groupby('grade')['recoveries'].agg(['count', 'mean', 'median']).round(2).reset_index()
display(recovery_by_grade) 

In [ ]:
fig = px.bar(recovery_by_grade.reset_index(), x='grade', y='mean', title='Average Recovery Amount by Loan Grade')
fig.show()

Observation:
* Grade G has the highest average recovery 1,933, compared with Grade A 789.
* Median recoveries also increase from A to G, so this pattern isn't driven only by a few unusually large recoveries.
* Higher-risk grades are associated with larger absolute recovery amounts among charged-off loans.

4. Recovery as a percentage of the original loan amount

In [ ]:
charged_off_df['recovery_pct'] = (charged_off_df['recoveries'] / charged_off_df['funded_amnt'] * 100)

In [ ]:
recovery_pct_by_grade = charged_off_df.groupby('grade')['recovery_pct'].agg(['mean', 'median']).round(2).reset_index()
display(recovery_pct_by_grade)

In [ ]:
fig = px.bar(recovery_pct_by_grade.reset_index(), x='grade', y='mean', title='Average Recovery-to-Loan Ratio by Grade')
fig.show()

Observation:

* Grade G has the highest average ratio at 9.45%, compared with 5.82% for Grade A.
* Both mean and median ratios increase across grades, indicating a consistent pattern in this dataset.
* Grade A's median of 1.88% is considerably lower than its mean of 5.82%, suggesting a right-skewed distribution.
* Higher-risk grades show greater recorded recoveries relative to original funded amounts among charged-off loans.

5. Recovery After Collection Fees

In [ ]:
charged_off_df['net_recovery'] = (charged_off_df['recoveries'] - charged_off_df['collection_recovery_fee'])
charged_off_df[['recoveries', 'collection_recovery_fee', 'net_recovery']].describe()

In [ ]:
net_recovery_by_grade = charged_off_df.groupby('grade')['net_recovery'].mean().round(2)

fig = px.bar(net_recovery_by_grade.reset_index(), x='grade', y='net_recovery', title='Average Net Recovery by Loan Grade')
fig.show()

Observation: 

* Average net recovery: Each charged-off loan generates approximately 978 in recorded recoveries after reported collection fees.
* Collection costs: Average collection fees represent approximately 16.6% of average gross recoveries.
* Skewed distribution: The median net recovery 460 is considerably lower than the mean 978, indicating that larger recoveries increase the average.
* Grade differences: Grade G has the highest average net recovery in absolute dollars, while Grade A has the lowest.

Business interpretation: Higher-risk grades generate larger net recovery amounts among charged-off loans, but this does not necessarily mean collections are more efficient for those grades. Loan exposure and time available for recovery may differ.

6. Net Recovery Relative to Loan Amount

In [ ]:
charged_off_df['net_recovery_pct'] = (charged_off_df['net_recovery'] / charged_off_df['funded_amnt'] * 100)

In [ ]:
net_recovery_pct_by_grade = charged_off_df.groupby('grade')['net_recovery_pct'].agg(['mean', 'median']).round(2).reset_index()
display(net_recovery_pct_by_grade)

In [ ]:
fig = px.bar(net_recovery_pct_by_grade.reset_index(), x='grade', y='mean', title='Average Net Recovery-to-Loan Ratio by Grade')
fig.show()

Observation:

* Grade A has the lowest average net recovery-to-loan ratio at 4.87%.
* Grade G has the highest at 7.91%.
* The ratio increases consistently from Grade A to Grade G.
* Higher-risk grades show greater net recoveries relative to original funded amounts among charged-off loans.
  
Business interpretation: Higher-risk loans show higher net recovery-to-original-loan ratios, but this doesn't necessarily mean better collection effectiveness. Differences in outstanding balances, repayment histories, and time available for recovery may influence the results.

7. Recovery by Loan Term

In [ ]:
recovery_by_term = charged_off_df.groupby('term')['net_recovery_pct'].agg(['count', 'mean', 'median']).round(2).reset_index()
display(recovery_by_term)

In [ ]:
fig = px.bar(recovery_by_term.reset_index(), x='term', y='mean', title='Average Net Recovery-to-Loan Ratio by Term')
fig.update_xaxes(type='category')
fig.show()

Observation:

* 60-month loans: Average net recovery ratio is 6.75%, compared with 5.70% for 36-month loans.
* Difference: 1.05 percentage points.
* Median: 60-month loans also have a higher median, suggesting the difference is not driven solely by unusually large recoveries.
  
Business interpretation: Longer-term charged-off loans show higher net recoveries relative to their original funded amounts. However, this doesn't establish better collection efficiency because loan grades, outstanding balances, and recovery timelines may differ.

8. Recovery by Loan Issue Year

In [ ]:
charged_off_df['issue_year'] = charged_off_df['issue_d'].dt.year

In [ ]:
recovery_by_year = charged_off_df.groupby('issue_year')['net_recovery_pct'].agg(['count', 'mean', 'median']).round(2).reset_index()
display(recovery_by_year)

In [ ]:
fig = px.line(recovery_by_year.reset_index(), x='issue_year', y='mean', markers=True, title='Average Net Recovery-to-Loan Ratio by Issue Year')
fig.show()

Observation:

Loans issued in 2017 or 2018 had much less time to become charged off and accumulate recoveries than loans issued in earlier years. This is particularly noticeable in your median values:
- 2016: 6.05%
- 2017: 0%
- 2018: 0%

A zero median means at least half of the charged-off loans in those issue-year groups had no recorded net recovery at the dataset snapshot. However, we cannot conclude that collection performance deteriorated in 2017–2018 without accounting for the time available to recover money.

9. Recovery Occurrence by Issue Year

In [ ]:
recovery_occurrence_year = (charged_off_df.groupby('issue_year')['has_recovery'].mean().mul(100).round(2).reset_index())

fig = px.line(recovery_occurrence_year, x='issue_year', y='has_recovery', markers=True, title='Recovery Occurrence (%) by Issue Year')
fig.show()

Observation:

Business interpretation
- Nearly all charged-off loans originated between 2007 and 2012 have recorded recoveries.
- Recovery occurrence decreases substantially for loans originated after 2013.
- The sharp decline for 2017–2018 loans is consistent with limited time available for collections, although the chart alone cannot establish the cause.
                                                                                 
We can not conclude that Organization's collection performance deteriorated. Recent loans had shorter observation periods, so recovery occurrence is not directly comparable across issue years.

10. Does Debt Settlement Affect Recovery?

In [ ]:
settlement_summary = charged_off_df.groupby('debt_settlement_flag').agg(loans=('recoveries', 'size'),recovery_occurrence=('has_recovery', 'mean'),avg_net_recovery_pct=('net_recovery_pct', 'mean')).round(3).reset_index()

settlement_summary['recovery_occurrence'] *= 100
display(settlement_summary.round(2))

In [ ]:
fig = px.bar(settlement_summary.reset_index(),x='debt_settlement_flag', y='avg_net_recovery_pct', title='Average Net Recovery-to-Loan Ratio by Settlement Status')
fig.show()

Observation:

* Charged-off loans flagged for debt settlement have an average net recovery ratio more than twice that of loans without a settlement flag.
* Debt settlement status is strongly associated with recorded net recoveries in this dataset.

However, we cannot conclude that debt settlement causes higher recovery. Settlement loans may differ in age, borrower risk, and collection history. We should also verify whether reported recoveries includes settlement payments before interpreting the financial relationship.

11. recovery occurrence by settlement status

In [ ]:
display(settlement_summary)

In [ ]:
fig = px.bar(settlement_summary.reset_index(),x='debt_settlement_flag', y='recovery_occurrence', title='Recovery Occurrence by Settlement Status')
fig.show()

Observation:

* Approximately 86% of settlement-flagged loans have positive recoveries, compared with 65% of loans without a settlement flag.
* Settlement-flagged loans also showed higher average net recovery-to-original-loan ratios in our previous analysis.

However, these results indicate association, not causation. Settlement arrangements may be recorded after collection activity has already begun.

12. Recovery by Settlement Status

In [ ]:
charged_off_df.groupby('settlement_status', dropna=False).agg(loans=('recoveries', 'size'), recovery_occurrence=('has_recovery', 'mean'), avg_net_recovery_pct=('net_recovery_pct', 'mean')).round(2).reset_index()

Observation:

1. Completed settlements show the strongest recovery outcomes.
- 92% of completed-settlement loans have positive recorded recoveries.
- Their average net recovery-to-original-loan ratio is 21.86%, compared with 5.17% for loans without recorded settlements.
- That's approximately 4.2 times higher.
2. Active and broken settlements show much lower net recovery ratios.
- ACTIVE: 6.01%
- BROKEN: 7.14%
- COMPLETE: 21.86%
3. Settlement completion is strongly associated with recovery outcomes.

However, we cannot conclude that completing a settlement causes higher recovery. Settlement status is an outcome of the collections process, and completed settlements are likely to have more recorded payments.

Also, the 21.86% represents net recovery relative to the original funded loan amount, not the outstanding balance at charge-off.

13. Settlement Amount vs. Recovery

In [ ]:
settled_df = charged_off_df[charged_off_df['settlement_status'].notna()].copy()

settled_df[['settlement_amount', 'recoveries', 'collection_recovery_fee']].describe()

In [ ]:
settled_df[['settlement_amount', 'recoveries']].corr()

Observation: 

Among 31,995 charged-off loans with settlement records, settlement amounts and recorded recoveries show a moderate positive correlation (r = 0.52). The average settlement amount is 5,052, compared with average recorded recoveries of 2,416. This suggests that larger settlements are generally associated with higher recoveries, although settlement obligations and realized collections may represent different financial measures.

### Borrower characteristics

In [ ]:
borrower_cols = ['annual_inc', 'dti', 'net_recovery_pct']

correlation = charged_off_df[borrower_cols].corr().round(2)
display(correlation)

In [ ]:
fig = px.imshow(correlation, text_auto=True, title='Borrower Characteristics vs Net Recovery')
fig.show()

In [ ]:
correlation['net_recovery_pct'].sort_values(ascending=False)

Observation:
- Annual income (r = 0.03): Higher-income borrowers do not consistently generate higher net recovery ratios.
- DTI (r = 0.01): Debt-to-income ratio has almost no linear association with the recovery ratio.
- Income vs. DTI (r = -0.13): A weak negative relationship exists between these two borrower characteristics.
These findings suggest that income and DTI alone are unlikely to explain much of the variation in recovery performance through a simple linear relationship.

Area	Main finding

* Recovery occurrence:	67.64% of charged-off loans have positive recoveries
* Gross recovery:	Average of 1172 per charged-off loan
* Net recovery:	Average of 978 after reported collection fees
* Loan grade:	Net recovery-to-loan ratio rises from 4.87% (A) to 7.91% (G)
* Loan term:	60-month loans average 6.75%, versus 5.70% for 36-month loans
* Origination year:	Recent cohorts show lower recorded recoveries, potentially due to limited observation time
* Settlement completion:	Completed settlements have a 21.86% average net recovery-to-loan ratio
* Annual income and DTI:	Negligible linear correlations with net recovery ratio

In [ ]:
charged_off_df['loan_amount_group'] = pd.cut( charged_off_df['loan_amnt'],bins=[0, 5000, 10000, 15000, 25000, 40000],labels=['0–5K', '5–10K', '10–15K', '15–25K', '25–40K'])

recovery_by_amount = charged_off_df.groupby('loan_amount_group', observed=True)['net_recovery_pct'].agg(['count', 'mean', 'median']).round(2).reset_index()

display(recovery_by_amount)

In [ ]:
fig = px.bar(recovery_by_amount,x='loan_amount_group', y='mean', title='Average Net Recovery-to-Loan Ratio by Loan Amount')
fig.show()

In [ ]:
charged_off_df[['out_prncp', 'funded_amnt', 'total_rec_prncp', 'recoveries']].describe()

Key finding: Larger charged-off loans generally have higher net recovery-to-original-loan ratios.
- Loans above $25K have the highest average ratio at 6.76%.
- Loans between $5K and $10K have the lowest at 5.69%.
- The median ratio also increases across the loan-size groups.
However, this does not prove that larger loans are easier to recover.

### Final business conclusions

1. Recovery occurrence declines across more recent origination cohorts. However, these cohorts had shorter observation periods, so differences in recorded recoveries cannot be interpreted as changes in collection effectiveness without adjusting for recovery time.

2. Loan grade and loan term are associated with differences in recovery outcomes. However, these differences may be affected by original loan sizes, remaining balances, and collection timelines.

3. Completed settlements are associated with substantially higher recorded recovery ratios. This is descriptive evidence, not proof that settlement completion causes improved recoveries.

4. Borrower income and DTI have weak linear associations with recovery ratios. Additional features or nonlinear analysis may be necessary to explain variations in collections.

5. Limitations to document: Our ratios use original funded amounts rather than balances outstanding at charge-off. We also lack confirmed charge-off and recovery transaction dates, so recovery timing cannot be consistently controlled. Settlement status is recorded after origination, and the reported collection fees may not capture all collection expenses

6. Loan size and recovery outcomes:Larger charged-off loans generally show higher net recovery-to-original-funded-amount ratios. Loans between $25,000 and 40,000 have an average ratio of 6.76%, compared with 5.69% for loans between 5,000 and 10,000. However, this relationship does not establish that larger loans are easier to recover, because borrower risk, collection timing, and unpaid balances may differ.

#### Vintage Analysis

Vintage analysis groups loans by the period in which they originated and tracks their performance as they age.

In [ ]:
df['issue_month'] = df['issue_d'].dt.to_period('M').astype(str)

vintage_df = (df.groupby('issue_month').agg(total_loans=('loan_status', 'size'),charged_off=('is_charged_off', 'sum')).reset_index())

vintage_df['chargeoff_pct'] = (vintage_df['charged_off'] / vintage_df['total_loans'] * 100).round(2)

vintage_df.head(10)

In [ ]:
vintage_df[['total_loans', 'chargeoff_pct']].describe()

In [ ]:
fig = px.line(vintage_df,x='issue_month',y='chargeoff_pct',title='Observed Charge-Off Percentage by Origination Month',labels={'issue_month': 'Origination Month','chargeoff_pct': 'Charge-Off Percentage (%)'})

fig.show()

Observation:

* 2007–2008: Charge-offs fluctuate between 11–32%, highlighting the need to monitor smaller cohorts separately.
* 2009–2013: Observed charge-offs generally range from 10–19%, providing historical context for portfolio risk.
* 2014–2016: Relatively stable charge-offs of 15–19%, useful for identifying changes in lending performance.
* 2017–2018: Declining charge-offs largely reflect shorter observation periods, making direct comparisons misleading.

Business Impact: Cohort monitoring helps credit risk teams identify potential portfolio deterioration, investigate underwriting changes, and prioritize risk reviews. However, loan-age differences must be accounted for before making lending policy or loss forecasting decisions.

**The Key issue with this dataset are:**

1. Outcome maturity: Many 2017–2018 loans were still active when the dataset was collected.
2. Selection bias: Excluding active loans means we model resolved outcomes, not the complete lending population.
3. Missing charge-off dates: We cannot reliably define defaults within a fixed 12-, 24-, or 36-month horizon.

Model Feasibility Assessment is a preliminary check performed before building a predictive model to determine whether the available data can reliably support the business objective.

A model feasibility assessment was conducted to examine loan outcome maturity by origination year and term. Based on the results, the predictive model was restricted to resolved 36-month loans originated between 2007 and 2015, with chronological training, validation, and test periods.

In [ ]:
feasibility = pd.crosstab(
    [df['issue_d'].dt.year, df['term']],
    df['loan_status']
)[['Fully Paid', 'Charged Off', 'Current']]

feasibility['total'] = feasibility.sum(axis=1)

feasibility['unresolved_pct'] = (
    feasibility['Current'] / feasibility['total'] * 100
).round(2)

display(feasibility)

Observations:
* 2007–2013: Historical cohorts are largely resolved, although we still need to account for other loan statuses.
* 2014–2015: 36-month loans are almost entirely resolved, while many 60-month loans remain active.
* 2016–2018: Large unresolved populations make simple Fully Paid versus Charged Off comparisons unsuitable for estimating default risk across all originated loans.

**Model feasibility:** Loan outcome maturity varies substantially by origination year and term. By 2015, nearly all 36-month loans had resolved outcomes in the primary status categories, whereas newer cohorts and 60-month loans had significant unresolved balances. The primary model will therefore focus on historical 36-month originations, using chronological validation to reduce maturity-related bias.

I am selecting 36-month loans from 2007–2015 because they have had enough time to reach an outcome, allowing us to study charge-off risk more fairly. I would not include 60-month loans in the primary model because their outcome maturity differs substantially across the potential validation years. They can be considered in a separate analysis later.

In [ ]:
pd.crosstab(df.loc[(df['term'] == 36) & (df['issue_d'].dt.year <= 2015),'issue_d'].dt.year, df.loc[(df['term'] == 36) & (df['issue_d'].dt.year <= 2015),'loan_status'])

Loan population: 36-month loans originated during 2007–2015.

Target variable:
- 0 = Fully Paid, including special-policy Fully Paid
- 1 = Charged Off, including special-policy Charged Off
Exclude: Current, In Grace Period, and Late loans because their final outcomes are unknown.

Training: 2007–2013

Validation: 2014

Testing: 2015

In [ ]:
model_df = df[(df['term'] == 36) & (df['issue_d'].dt.year.between(2007, 2015)) & (df['loan_status'].str.contains('Fully Paid|Charged Off', na=False))].copy()

model_df['target'] = (model_df['loan_status'].str.contains('Charged Off')).astype(int)

model_df['issue_year'] = model_df['issue_d'].dt.year

model_df.groupby('issue_year')['target'].agg(['count', 'mean']).round(3).reset_index()

In [ ]:
features = [ 'grade','loan_amnt','int_rate','annual_inc','dti','home_ownership','verification_status','purpose','emp_length','delinq_2yrs','inq_last_6mths','revol_util']

In [ ]:
model_df[features].isna().mean().mul(100).round(2).sort_values(ascending=False)

In [ ]:
X = model_df[features].copy()
y = model_df['target']

train = model_df['issue_year'] <= 2013
val = model_df['issue_year'] == 2014
test = model_df['issue_year'] == 2015

X_train, y_train = X.loc[train], y.loc[train]
X_val, y_val = X.loc[val], y.loc[val]
X_test, y_test = X.loc[test], y.loc[test]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

preprocessing
- Median imputation for missing numerical values.
- Standardization for numerical features.
- One-hot encoding for categorical features.
- Train-only capping of extreme numerical values.

In [ ]:
class OutlierCapper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.upper_ = np.nanpercentile(X, 99.9, axis=0)
        return self

    def transform(self, X):
        return np.minimum(np.asarray(X, dtype=float), self.upper_)

In [ ]:
categorical = ['grade', 'home_ownership', 'verification_status','purpose', 'emp_length']

numerical = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'revol_util']

In [ ]:
preprocessor = ColumnTransformer([
    ('cat', make_pipeline(
        SimpleImputer(strategy='most_frequent'),
        OneHotEncoder(handle_unknown='ignore', drop='first')
    ), categorical),

    ('num', make_pipeline(
        OutlierCapper(),
        SimpleImputer(strategy='median'),
        StandardScaler()
    ), numerical)
])

In [ ]:
model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000)
)

model.fit(X_train, y_train)

In [ ]:
y_val_prob = model.predict_proba(X_val)[:, 1]

print("ROC-AUC:", round(roc_auc_score(y_val, y_val_prob), 3))
print("PR-AUC:", round(average_precision_score(y_val, y_val_prob), 3))
print("Charge-off prevalence:", round(y_val.mean(), 3))

**Risk Decile Analysis**

Banks use risk segmentation to examine whether higher model scores correspond to higher observed credit losses. We'll divide the 2014 validation loans into ten groups based on their predicted charge-off probabilities.

In [ ]:
risk_deciles = pd.DataFrame({
    'actual': y_val.to_numpy(),
    'predicted_prob': y_val_prob
})

risk_deciles['decile'] = pd.qcut(
    risk_deciles['predicted_prob'].rank(method='first'),
    10,
    labels=range(1, 11)
)

decile_summary = risk_deciles.groupby('decile', observed=True).agg(
    loans=('actual', 'size'),
    chargeoff_rate=('actual', 'mean'),
    avg_predicted_risk=('predicted_prob', 'mean')
)

decile_summary[['chargeoff_rate', 'avg_predicted_risk']] *= 100

display(decile_summary.round(2))

Observation: Risk Decile Analysis 

* Observed charge-off rates increase consistently from 4.31% in the lowest-risk decile to 27.53% in the highest-risk decile, representing approximately 6.39 times higher observed charge-off risk. This demonstrates meaningful risk differentiation across the validation portfolio.
* Predicted probabilities generally underestimate observed charge-off rates, indicating potential calibration issues.

Business Impact: Risk segmentation can support portfolio monitoring and targeted credit-risk reviews. Further validation is required before using model scores for lending decisions.

**The Brier score** measures probability prediction error. lower is better, but it must be compared with a baseline to judge whether it's useful.
Before selecting a classification threshold, we should check how closely predicted charge-off probabilities match actual outcomes.

In [ ]:
print("Brier Score:", round(brier_score_loss(y_val, y_val_prob), 4))
print("Average Predicted Risk:", round(y_val_prob.mean(), 4))
print("Actual Charge-Off Rate:", round(y_val.mean(), 4))

**Probability Calibration — Key Insights**

- Average predicted charge-off risk (12.61%) is below the observed rate (13.73%), indicating mild overall underprediction.
- The model achieved a Brier score of 0.1137 on the 2014 validation cohort, compared with approximately 0.1184 for a constant-probability baseline. However, the average predicted charge-off risk remained below the observed rate, indicating that probability calibration requires further assessment.

**Business Impact:** Better-calibrated probabilities can support more reliable credit-loss forecasting and risk-based portfolio monitoring.

**Precsion and recall at different thresholds using the 2014 validation dataset**

In [ ]:
results = []

for threshold in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    pred = (y_val_prob >= threshold).astype(int)

    results.append({
        'threshold': threshold,
        'precision': precision_score(y_val, pred, zero_division=0),
        'recall': recall_score(y_val, pred, zero_division=0),
        'f1': f1_score(y_val, pred, zero_division=0),
        'flagged_pct': pred.mean() * 100
    })

pd.DataFrame(results).round(3)

Classification Threshold Analysis — Key Insights

* 0.10 threshold: Captures 80% of charged-off loans but flags 60% of the portfolio.
* 0.15 threshold: Achieves the highest tested F1-score (0.31), with 52% recall and 31% of loans flagged.
* 0.20 threshold: Flags only 13% of loans, but recall falls to 26%.
* 0.50 threshold: Misses nearly all charged-off loans, making the default threshold unsuitable for charge-off detection.

Business Impact: Threshold selection involves balancing charge-off detection against false positives and review capacity. Risk-based segmentation is more appropriate until financial costs and operational constraints are established.

In [ ]:
pre = model.named_steps['columntransformer']

cat_names = pre.named_transformers_['cat'].named_steps[
    'onehotencoder'
].get_feature_names_out(categorical)

feature_names = list(cat_names) + numerical

coefficients = model.named_steps['logisticregression'].coef_[0]

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients,
    'odds_ratio': np.exp(coefficients)
})

display(coef_df.sort_values('coefficient', ascending=False).round(3))

Coefficient Analysis

* Higher interest rates, debt-to-income ratios, and recent credit inquiries are associated with increased modeled charge-off risk.
* Higher annual income is associated with lower modeled charge-off risk.
* Loan grade and purpose contribute to risk differentiation, but individual categorical coefficients require cautious interpretation.
* Correlation between lender-assigned grades and interest rates may affect coefficient stability.

Business Impact: The model identifies borrower and loan characteristics associated with historical charge-offs, supporting risk segmentation and portfolio monitoring rather than standalone lending decisions.

In [ ]:
encoder = model.named_steps['columntransformer'].named_transformers_['cat'].named_steps['onehotencoder']

print("Drop setting:", encoder.drop)
print("Total features:", model.named_steps['logisticregression'].coef_.shape[1])

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

y_test_prob = model.predict_proba(X_test)[:, 1]

print("Test Loans:", len(y_test))
print("Actual Charge-Off Rate:", round(y_test.mean() * 100, 2), "%")
print("ROC-AUC:", round(roc_auc_score(y_test, y_test_prob), 3))
print("PR-AUC:", round(average_precision_score(y_test, y_test_prob), 3))
print("Brier Score:", round(brier_score_loss(y_test, y_test_prob), 4))
print("Average Predicted Risk:", round(y_test_prob.mean() * 100, 2), "%")

In [ ]:
test_deciles = pd.DataFrame({
    'actual': y_test.to_numpy(),
    'predicted_prob': y_test_prob
})

test_deciles['decile'] = pd.qcut(
    test_deciles['predicted_prob'].rank(method='first'),
    10,
    labels=range(1, 11)
)

test_summary = test_deciles.groupby(
    'decile', observed=True
).agg(
    loans=('actual', 'size'),
    chargeoff_rate=('actual', 'mean'),
    avg_predicted_risk=('predicted_prob', 'mean')
)

test_summary[['chargeoff_rate', 'avg_predicted_risk']] *= 100

display(test_summary.round(2))

In [ ]:
# Use the existing model's 2014 predicted probabilities

y_val_prob = model.predict_proba(X_val)[:, 1]

# Fit a simple calibration model
calibrator = LogisticRegression()

calibrator.fit(
    y_val_prob.reshape(-1, 1),
    y_val
)

In [ ]:
# Original 2015 predictions

y_test_prob = model.predict_proba(X_test)[:, 1]

# Calibrated 2015 predictions
y_test_calibrated = calibrator.predict_proba(
    y_test_prob.reshape(-1, 1)
)[:, 1]

In [ ]:
comparison = pd.DataFrame({
    'Metric': [
        'ROC-AUC',
        'Brier Score',
        'Average Predicted Risk (%)',
        'Actual Charge-Off Rate (%)'
    ],
    'Original': [
        roc_auc_score(y_test, y_test_prob),
        brier_score_loss(y_test, y_test_prob),
        y_test_prob.mean() * 100,
        y_test.mean() * 100
    ],
    'Calibrated': [
        roc_auc_score(y_test, y_test_calibrated),
        brier_score_loss(y_test, y_test_calibrated),
        y_test_calibrated.mean() * 100,
        y_test.mean() * 100
    ]
})

display(comparison.round(4))

In [ ]:
from sklearn.calibration import calibration_curve

actual_orig, pred_orig = calibration_curve(
    y_test, y_test_prob, n_bins=10, strategy='quantile'
)

actual_cal, pred_cal = calibration_curve(
    y_test, y_test_calibrated, n_bins=10, strategy='quantile'
)

plt.figure(figsize=(7, 5))

plt.plot([0, 1], [0, 1], '--', label='Perfect Calibration')

plt.plot(
    pred_orig, actual_orig,
    marker='o', label='Original'
)

plt.plot(
    pred_cal, actual_cal,
    marker='o', label='Calibrated'
)

plt.xlabel('Average Predicted Charge-Off Risk')
plt.ylabel('Actual Charge-Off Rate')
plt.title('2015 Test: Probability Calibration')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
print("Original Brier:",
      round(brier_score_loss(y_test, y_test_prob), 6))

print("Calibrated Brier:",
      round(brier_score_loss(y_test, y_test_calibrated), 6))

#### Post-hoc Probability Recalibration — Key Findings

- Recalibration slightly improved the Brier score from **0.121513 to 0.121378**, indicating a small improvement in prediction accuracy.
- Average predicted charge-off risk increased from **11.60% to 12.75%**, moving closer to the actual rate of **14.87%**.
- ROC-AUC remained **0.679**, showing that the model's ability to rank risk did not change.
- **Conclusion:** Recalibration helped reduce risk underestimation, but the model is still better at identifying higher-risk loans than predicting exact charge-off probabilities.

*Note: This was a post-hoc experiment using 2014 data for calibration and 2015 data for evaluation.*

## Final Credit Risk Model — Business Insights

* Achieved 0.679 ROC-AUC on 282,853 loans in the 2015 test cohort.
* Charge-off rates ranged from 4.50% to 30.75% across risk deciles, a 6.83× difference.
* The highest-risk 10% of loans accounted for 20.7% of total charge-offs.
* Recalibration slightly improved the Brier score (0.121513 to 0.121378), but predicted risk remained below actual charge-offs.

Business Implication: The model helps identify higher-risk loans for portfolio monitoring but requires further validation before use in lending decisions.

### Model Limitations

* Selection bias: Only resolved 36-month loans from 2007–2015 were included, potentially excluding unresolved outcomes.
* Historical differences: Organization's underwriting and pricing policies may have changed over time.
* Lender-assigned features: Grade and interest rate reflect Organization's existing risk assessments.
* Calibration: Recalibration improved average predicted risk from 11.60% to 12.75%, but it remained below the actual 14.87% charge-off rate.
* Applicability: The model supports historical risk segmentation, not automated lending decisions or reliable individual default probability estimates.